In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
from PIL import Image
from pathlib import Path

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
figures_dir = base_path / "dphil_paper_2/results/figures"
output_dir = base_path / "dphil_paper_2/results"

In [ ]:
# Point these to your exported maps (PNG). Order = left→right
img_paths = [
    # figures_dir / "avoided_fluvial_ead_300m_linear_USD_plain_figure_3a.png",
    output_dir / "figures/avoided_fluvial_ead_300m_linear_USD_plain_figure_3a.png",
    # figures_dir / "forest_avoided_damages_linear_USD_plain_units.png",
    output_dir / "figures/Fig_3b_forest_avoided_damages_linear_USD_plain_units.png",
    figures_dir / "Fig3c_avoided_EAD_max_USD_millions_linear.png",
    figures_dir / "avoided_share_pct_by_catchment_map.png",

]






In [ ]:
panel_labels = ["a", "b"]          # panel letters
panel_titles = [None, None]        # or short titles per panel (strings or None)

In [ ]:
img_paths = img_paths[:2]  # <-- just the first two

# --- Figure size ---------------------------------------------------------------
mm = 1/25.4
width_mm   = 180
panel_h_mm = 55
N = len(img_paths)
fig_h_mm   = panel_h_mm * N + 8

mpl.rcParams.update({
    "savefig.dpi": 600,
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial"],
})

# --- Crop helper ---------------------------------------------------------------
def autocrop_white(im: Image.Image, tol=5) -> Image.Image:
    arr = np.asarray(im)
    if arr.ndim == 3 and arr.shape[2] == 4:  # RGBA
        rgb = arr[..., :3]
        alpha = arr[..., 3] > 0
        mask = (np.any(rgb < 255 - tol, axis=2)) | alpha
    else:
        rgb = arr[..., :3] if arr.ndim == 3 else np.repeat(arr[..., None], 3, axis=2)
        mask = np.any(rgb < 255 - tol, axis=2)
    if not mask.any():
        return im
    ys, xs = np.where(mask)
    box = (xs.min(), ys.min(), xs.max()+1, ys.max()+1)
    return im.crop(box)

# --- Load images ---------------------------------------------------------------
images = []
for p in img_paths:
    p = Path(p)
    if not p.exists():
        print(f"Warning: not found → {p}")
        continue
    im = Image.open(p).convert("RGBA")
    im = autocrop_white(im, tol=4)
    images.append(im)
if not images:
    raise FileNotFoundError("None of the image paths exist. Check `img_paths`.")

# --- Labels/titles -------------------------------------------------------------
panel_labels = ["a", "b"][:N]
panel_titles = [None] * N  # or e.g. ["Avoided EAD (disc.)", "Total cost"]

# --- Render --------------------------------------------------------------------
fig, axes = plt.subplots(N, 1, figsize=(width_mm*mm, fig_h_mm*mm), constrained_layout=False)
if N == 1:
    axes = np.array([axes])

for ax, im, label, title in zip(axes, images, panel_labels, panel_titles):
    ax.imshow(im)
    ax.axis("off")
    ax.text(0.01, 0.99, f"({label})", transform=ax.transAxes,
            ha="left", va="top", fontsize=7, fontweight="bold", color="black")
    if title:
        ax.text(0.5, 0.01, title, transform=ax.transAxes,
                ha="center", va="bottom", fontsize=6.5)

fig.subplots_adjust(left=0.02, right=0.98, top=0.98, bottom=0.02, hspace=0.06)


# # --- Save ---------------------------------------------------------------------
# # use your existing out_dir if defined; otherwise default to "<cwd>/figures"
# panel_png = out_dir / "panel_top2_vertical.png"

# fig.savefig(panel_png, bbox_inches="tight", facecolor="white")
# plt.show()
# print("Saved:", panel_png)


# ---- Save outputs ------------------------------------------------------------
fig_3_path = figures_dir / "Fig3_panels_row"
fig.savefig(fig_3_path.with_suffix(".png"), bbox_inches="tight", facecolor="white")
plt.show()
print("Saved:", fig_3_path.with_suffix(".png"))

In [ ]:
# # Figure size — same width for all panels; tune panel_h_mm to taste
# mm = 1/25.4
# width_mm   = 180
# panel_h_mm = 55                      # height of each panel
# fig_h_mm   = panel_h_mm * 3 + 8      # add a little margin

# # ---- Ensure Arial (matches your maps) ---------------------------------------
# mpl.rcParams.update({
#     "savefig.dpi": 600,
#     "font.family": "sans-serif",
#     "font.sans-serif": ["Arial"],
# })

# # ---- (optional) auto-crop white margins around each PNG ---------------------
# def autocrop_white(im: Image.Image, tol=5) -> Image.Image:
#     """Trim near-white margins; keeps transparent pixels if present."""
#     arr = np.asarray(im)
#     if arr.ndim == 3 and arr.shape[2] == 4:  # RGBA
#         rgb = arr[..., :3]
#         alpha = arr[..., 3] > 0
#         mask = (np.any(rgb < 255 - tol, axis=2)) | alpha
#     else:
#         rgb = arr[..., :3] if arr.ndim == 3 else np.repeat(arr[..., None], 3, axis=2)
#         mask = np.any(rgb < 255 - tol, axis=2)
#     if not mask.any():
#         return im
#     ys, xs = np.where(mask)
#     box = (xs.min(), ys.min(), xs.max()+1, ys.max()+1)
#     return im.crop(box)



In [ ]:
# # ---- Load images -------------------------------------------------------------
# images = []
# for p in img_paths:
#     im = Image.open(Path(p)).convert("RGBA")
#     im = autocrop_white(im, tol=4)  # set to None to disable
#     images.append(im)
